In [ ]:
import sys
import pandas as pd
import os
import dotenv

dotenv.load_dotenv(dotenv_path='/Users/mlchrzan/Library/CloudStorage/OneDrive-Personal/Professional/Data Science/Maryland/bechmark analysis pipeline/benchmark-analysis/.env')

sys.path.append('/Users/mlchrzan/Library/CloudStorage/OneDrive-Personal/Professional/Data Science/Personal Projects/pairadigm')
import pairadigm as pdgm

# Combine Human Annotations

In [ ]:
import glob
import re

# Initialize dataframes for each concept
valence_df = None
arousal_df = None
dominance_df = None

# Get all annotation files
files = glob.glob('Data/human annotations/Completed/pairs_for_human_annotation - A*.xlsx')

# Process each file
for file in files:
    # Extract annotator number from filename
    match = re.search(r'A(\d+)', file)
    annotator_num = match.group(1)
    
    # Read all sheets
    sheets = pd.read_excel(file, sheet_name=None)
    
    # Process each concept sheet
    for sheet_name, df in sheets.items():
        concept = sheet_name.lower()
        
        # Skip if not a concept sheet
        if concept not in ['valence', 'arousal', 'dominance']:
            continue
        
        # Create decision column name
        decision_col = f'A{annotator_num}'
        
        # Rename concept column and map values
        df_copy = df.copy()
        # Make the fifth column name (the prompt given to the annotators) the name of the concept
        df_copy.rename(columns={df_copy.columns[4]: concept}, inplace=True)
        df_copy[decision_col] = df_copy[concept].map({1: 'Text1', 2: 'Text2', 0: 'Tie'})

        # Adjust the Notes column to be a string and rename it with the annotator_num
        if 'Notes' in df_copy.columns:
            df_copy['Notes'] = df_copy['Notes'].astype(str)
            df_copy['Notes'] = df_copy['Notes'].replace('nan', '')
            df_copy.rename(columns={'Notes': f'Notes_A{annotator_num}'}, inplace=True)
            notes_col = f"Notes_A{annotator_num}"
        
        # Keep only necessary columns
        base_cols = ['item1', 'item2', 'text1', 'text2']
        df_copy = df_copy[base_cols + [decision_col] + [notes_col]]
        
        # Merge with appropriate concept dataframe
        if concept == 'valence':
            if valence_df is None:
                valence_df = df_copy
            else:
                valence_df = valence_df.merge(df_copy, on=base_cols, how='outer')
        elif concept == 'arousal':
            if arousal_df is None:
                arousal_df = df_copy
            else:
                arousal_df = arousal_df.merge(df_copy, on=base_cols, how='outer')
        elif concept == 'dominance':
            if dominance_df is None:
                dominance_df = df_copy
            else:
                dominance_df = dominance_df.merge(df_copy, on=base_cols, how='outer')

print(f"Valence annotations: {valence_df.shape}")
print(f"Arousal annotations: {arousal_df.shape}")
print(f"Dominance annotations: {dominance_df.shape}")

# Prep pairadigms

In [ ]:
# Load CGCOT Prompts
with open('Data/cgcot_prompts/valence.txt', 'r') as file:
    val_prompts = file.read().splitlines()

with open('Data/cgcot_prompts/arousal.txt', 'r') as file:
    aro_prompts = file.read().splitlines()

with open('Data/cgcot_prompts/dominance.txt', 'r') as file:
    dom_prompts = file.read().splitlines()

In [ ]:
# Initialize Pairadigm objects
p_val = pdgm.Pairadigm(
    data=valence_df,
    item_id_cols=['item1', 'item2'],
    item_text_cols=['text1', 'text2'],
    paired=True,
    annotated=True,
    annotator_cols=[col for col in valence_df.columns if col.startswith('A')],
    cgcot_prompts=val_prompts,
    model_name=["gpt-5", "claude-sonnet-4-5-20250929"],
    api_key=[os.getenv('OPENAI_API_KEY'), os.getenv('ANTHROPIC_API_KEY')],
    target_concept='valence'
)

p_aro = pdgm.Pairadigm(
    data=arousal_df,
    item_id_cols=['item1', 'item2'],
    item_text_cols=['text1', 'text2'],
    paired=True,
    annotated=True,
    annotator_cols=[col for col in arousal_df.columns if col.startswith('A')],
    cgcot_prompts=aro_prompts,
    model_name=["gpt-5", "claude-sonnet-4-5-20250929"],
    api_key=[os.getenv('OPENAI_API_KEY'), os.getenv('ANTHROPIC_API_KEY')],
    target_concept='arousal'
)

p_dom = pdgm.Pairadigm(
    data=dominance_df,
    item_id_cols=['item1', 'item2'],
    item_text_cols=['text1', 'text2'],
    paired=True,
    annotated=True,
    annotator_cols=[col for col in dominance_df.columns if col.startswith('A')],
    cgcot_prompts=dom_prompts,
    model_name=["gpt-5", "claude-sonnet-4-5-20250929"],
    api_key=[os.getenv('OPENAI_API_KEY'), os.getenv('ANTHROPIC_API_KEY')],
    target_concept='dominance'
)

# Generate Annotations

In [ ]:
for p in [p_val, p_aro, p_dom]:
    p.generate_breakdowns_from_paired()
    p.generate_pairwise_annotations(allow_ties=True)

In [ ]:
p_dom.pairwise_df

# IRR

In [ ]:
p_val.irr()

In [ ]:
p_aro.irr()

In [ ]:
p_dom.irr()

In [ ]:
irr = pd.DataFrame()
for p in [p_val, p_aro, p_dom]:
    concept_irr_df = p.irr()
    concept_irr_df['Concept'] = p.target_concept
    irr = pd.concat([irr, concept_irr_df], ignore_index=True)
irr

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Rename the groups for better readability
irr['group'] = irr['group'].replace({
    'human': 'Human Annotators',
    'llm': 'LLM Annotators',
    'all': 'All Annotators'
})

# Create a facet plot with separate subplots for each group
g = sns.catplot(
    data=irr,
    x='Concept',
    y='score',
    col='group',
    kind='bar',
    height=5,
    aspect=1.2,
    palette='Set2',
    hue='Concept',
    legend=False
)

# Customize the plot
g.set_axis_labels('Concept', 'IRR Score', fontsize=12)
g.set_titles(col_template='{col_name}', size=14)

# Add a horizontal line at different agreement thresholds
for ax in g.axes.flat:
    ax.axhline(y=0.8, color='red', linestyle='--', alpha=0.5, label='Substantial')
    ax.axhline(y=0.6, color='orange', linestyle='--', alpha=0.5, label='Moderate')
    ax.set_ylim(0, 1)

# Add caption for the horizontal lines
handles, labels = ax.get_legend_handles_labels()
handles += [plt.Line2D([0], [0], color='red', linestyle='--', alpha=0.5),
            plt.Line2D([0], [0], color='orange', linestyle='--', alpha=0.5)]
g._legend = g.fig.legend(handles, labels, title='Agreement Levels', loc='upper right', fontsize=10)

# Add overall title
plt.suptitle('Inter-Rater Reliability by Group and Concept', y=1.02, fontsize=16)
plt.tight_layout()
plt.show()

# Transitivity Check

In [ ]:
p_val.check_transitivity()

In [ ]:
p_aro.check_transitivity()

In [ ]:
p_dom.check_transitivity()

# Alt-Test

In [ ]:
p_val.plot_epsilon_sensitivity()

In [ ]:
p_aro.plot_epsilon_sensitivity()

In [ ]:
p_dom.plot_epsilon_sensitivity()

# Save pairadigms

In [ ]:
p_val.save('Results/pairadigms/valence_pairadigm_11-6-25.pkl')
p_aro.save('Results/pairadigms/arousal_pairadigm_11-6-25.pkl')
p_dom.save('Results/pairadigms/dominance_pairadigm_11-6-25.pkl')

# Valence Alt-Test Failure Exploration

In [ ]:
if not hasattr(pdgm.LLMClient, '_original_initialize_client'):
    pdgm.LLMClient._original_initialize_client = pdgm.LLMClient._initialize_client

def patched_initialize_client(self):
    # Ensure base_url exists before it's accessed
    if not hasattr(self, 'base_url'):
        self.base_url = None
    return self._original_initialize_client()

pdgm.LLMClient._initialize_client = patched_initialize_client

# Load the pairadigm
p_val = pdgm.load_pairadigm('Results/pairadigms/valence_pairadigm_11-6-25.pkl')
p_aro = pdgm.load_pairadigm('Results/pairadigms/arousal_pairadigm_11-6-25.pkl')
p_dom = pdgm.load_pairadigm('Results/pairadigms/dominance_pairadigm_11-6-25.pkl')

In [ ]:
for p in [p_val, p_aro, p_dom]:
    paired_results = p.pairwise_df
    paired_results.to_csv(f'Results/paired_results/{p.target_concept}_paired_results.csv', index=False)

In [ ]:
p_dom.dawid_skene_annotator_ranking(random_seed=1234)

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Define human and LLM annotators
human_annotators = [col for col in p_val.pairwise_df.columns if col.startswith('A')]
llm_annotators = [col for col in p_val.pairwise_df.columns if col.startswith('decision')]
all_annotators = human_annotators + llm_annotators

# Initialize a matrix to store agreement percentages
agreement_matrix = np.zeros((len(all_annotators), len(all_annotators)))

# Calculate agreement percentages
for i, llm_col in enumerate(all_annotators):
    for j, human_col in enumerate(all_annotators):
        # Calculate percentage of rows where values match
        matches = (p_val.pairwise_df[llm_col] == p_val.pairwise_df[human_col]).sum()
        total = len(p_val.pairwise_df)
        agreement_pct = (matches / total) * 100
        agreement_matrix[i, j] = agreement_pct

# Create clean labels by removing "decision_" prefix
clean_labels = [col.replace('decision_', '') for col in all_annotators]
clean_labels = [col.replace('-20250929', '') for col in clean_labels]

# Create a DataFrame for better labeling
agreement_df = pd.DataFrame(
    agreement_matrix,
    index=clean_labels,
    columns=clean_labels
)

# Create a mask for the upper triangle
import numpy as np
mask = np.triu(np.ones_like(agreement_df, dtype=bool), k=1)

# Plot the heatmap with colorblind-friendly palette
plt.figure(figsize=(10, 6))
sns.heatmap(
    agreement_df,
    annot=True,
    fmt='.1f',
    cmap='viridis',
    vmin=0,
    vmax=100,
    cbar_kws={'label': 'Agreement Percentage (%)'},
    linewidths=0.5,
    linecolor='gray',
    mask=mask
)
plt.title('%Agreement Between LLM and Human Annotators (Valence)', fontsize=14)
plt.tight_layout()
plt.show()